# Schelling Segregation (Mesa 3.5)

This notebook targets **Mesa 3.5** and uses its most modern idioms end to end. Randomness is seeded
through the `rng=` argument, agents are created in bulk with `create_agents`, and the space is the
stable `discrete_space` grid. To advance the model we call `model.run_for(n)`, the Mesa 3.5
run-control method that performs `n` model steps in a single call (equivalent to writing
`for _ in range(n): model.step()`) while advancing the universal `model.time` clock.

In [ ]:
# Environment pin (uncomment to install):
#!pip install -U "mesa[rec]==3.5.1"

In [ ]:
import mesa
import numpy as np
from mesa.discrete_space import CellAgent, OrthogonalMooreGrid

In [ ]:
class Household(CellAgent):
    def __init__(self, model, cell, kind):
        super().__init__(model)
        self.cell = cell
        self.kind = kind

    def is_happy(self):
        neighbours = self.cell.neighborhood.agents
        same = [a for a in neighbours if a.kind == self.kind]
        return len(same) >= 2

    def step(self):
        if not self.is_happy():
            self.cell = self.model.grid.select_random_empty_cell()

In [ ]:
class Schelling(mesa.Model):
    def __init__(self, n=50, width=10, height=10, rng=None):
        super().__init__(rng=rng)
        self.grid = OrthogonalMooreGrid((width, height), torus=True, random=self.random)
        Household.create_agents(
            self, n,
            cell=self.random.choices(self.grid.all_cells.cells, k=n),
            kind=self.random.choices([0, 1], k=n),
        )
        self.datacollector = mesa.DataCollector(
            model_reporters={
                "share_happy": lambda m: float(np.mean([a.is_happy() for a in m.agents]))
            }
        )

    def step(self):
        self.agents.shuffle_do("step")
        self.datacollector.collect(self)
        # Example AgentSet access (Mesa 3.5): to_list() for positional indexing.
        first = self.agents.to_list()[0]
        _ = (first.kind, self.time)

In [ ]:
model = Schelling(rng=42)
model.run_for(20)

print(f"After {model.steps} steps, model.time = {model.time}.")

In [ ]:
results = mesa.batch_run(
    Schelling,
    parameters={"n": [20, 40]},
    rng=[1, 2, 3],
    max_steps=20,
)
print(f"Collected {len(results)} runs.")

## Visualization

The renderer uses the Mesa 3.5 `setup_agents` chain on `SpaceRenderer`, styling each household by its
kind with an `AgentPortrayalStyle`.

In [ ]:
from mesa.visualization import SolaraViz, SpaceRenderer, make_plot_component
from mesa.visualization.components import AgentPortrayalStyle


def agent_portrayal(agent):
    color = "tab:blue" if agent.kind == 0 else "tab:orange"
    return AgentPortrayalStyle(color=color, marker="o", size=30)


renderer = SpaceRenderer(Schelling(), backend="matplotlib").setup_agents(agent_portrayal)
page = SolaraViz(Schelling(), renderer, components=[], name="Schelling")